# Chapter 3 - Matching the Dispersion Suppressor to the Straight Section

## Introduction

The periodic betas in the arcs are different from the periodic betas in the straight sections. After suppressing the dispersion to zero, four quadrupoles after the dispersion suppressor are needed to match $\alpha_a$, $\alpha_b$, $\beta_a$, and $\beta_b$ to the periodic betas in the following straight section.

![Figure 1: Dispersion suppressor match section.](assets/chapter3_dispersion_suppressor_match_section.png)

## Goal

We build the Chapter 3 matching section from the optimized forward dispersion suppressor obtained in Chapter 2.

Chapter 2 saved the optimized strengths in

```text
lattices/chapter_2/chapter2_dispsupF_solution.jl
```

This notebook reads that file, rebuilds the Chapter 2 forward dispersion suppressor `DISPSUPF`, and then adds the Chapter 3 matching section:

```text
DISPSUPF → MSSF
```

The matching section `MSSF` contains four new quadrupoles:

```text
QFF2, QDF2, QFF3, QDF3
```

Their strengths are optimized so that the Twiss parameters at the end of the line match the periodic Twiss parameters of the straight section ahead in the beam direction:

$$
(\beta_a,\alpha_a,\beta_b,\alpha_b)_\mathrm{END}
=
(\beta_a,\alpha_a,\beta_b,\alpha_b)_\mathrm{SS}.
$$


## Workflow

```text
read lattices/chapter_2/chapter2_dispsupF_solution.jl
→ rebuild Chapter 2 DISPSUPF
→ compute the Twiss at the end of DISPSUPF
→ add DB
→ add QFF2, QDF2, QFF3, QDF3
→ build MSSF
→ compute matching residuals
→ compute the residual Jacobian with GTPSA parameter derivatives
→ solve for the four matching strengths
→ export Chapter 3 strengths
```


## Packages


In [1]:
using SciBmad
using GTPSA
using DifferentiationInterface
import DifferentiationInterface as DI

using LinearAlgebra
using Printf


## 3.1 Read the Chapter 2 solution file

Chapter 2 exported the optimized strengths to

```text
lattices/chapter_2/chapter2_dispsupF_solution.jl
```

This file should define

```julia
kQF_arc
kQD_arc
kQFF1
kQDF1
```

We load these values into the current notebook.


In [2]:
solution_path = joinpath("lattices", "chapter_2", "chapter2_dispsupF_solution.jl")

if isfile(solution_path)
    include(solution_path)

    println("Loaded Chapter 2 solution:")
    @show kQF_arc
    @show kQD_arc
    @show kQFF1
    @show kQDF1
else
    error("Cannot find $(solution_path). Run Chapter 2 first.")
end


Loaded Chapter 2 solution:
kQF_arc = 0.3126590970351496
kQD_arc = -0.3127928794106306
kQFF1 = 0.31293818192092665
kQDF1 = -0.31531898515008894


-0.31531898515008894

## 3.2 Define shared lattice elements

The arc and dispersion-suppressor geometry uses

```text
D1: 0.609 m
D2: 1.241 m
B : full-strength bend
BH: half-strength bend
```

The Chapter 3 matching section additionally uses

```text
DB: 5.855 m
```


In [3]:
L_quad = 0.5

D1_len = 0.609
D2_len = 1.241
DB_len = 5.855

B_len = 6.86
B_angle = π / 132
BH_angle = B_angle / 2

species_ref = Species("electron")
E_ref = 18e9

D1() = Drift(L=D1_len)
D2() = Drift(L=D2_len)
DB() = Drift(L=DB_len)

B()  = SBend(L=B_len, angle=B_angle)
BH() = SBend(L=B_len, angle=BH_angle)


BH (generic function with 1 method)

## 3.3 Rebuild the Chapter 1 arc FODO cell

We use the optimized arc quadrupole strengths saved by Chapter 2:

```julia
kQF_arc, kQD_arc
```

The forward arc FODO cell is

```text
QF, D1, B, D2, QD, D1, B, D2
```

We use this cell to compute the periodic arc Twiss parameters at the entrance of the dispersion suppressor.


In [4]:
QF_arc() = Quadrupole(L=L_quad, Kn1=kQF_arc)
QD_arc() = Quadrupole(L=L_quad, Kn1=kQD_arc)

function build_FODOAF()
    return Beamline(
        [QF_arc(), D1(), B(), D2(), QD_arc(), D1(), B(), D2()],
        species_ref=species_ref,
        E_ref=E_ref,
    )
end

fodoAF = build_FODOAF()


Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1              Quadrupole   0
  2              Drift        0.5
  3              SBend        1.109
  4              Drift        7.969
  5              Quadrupole   9.21
  6              Drift        9.71
  7              SBend        10.319
  8              Drift        17.179


## 3.4 Rebuild the Chapter 2 dispersion suppressor

Chapter 2 optimized the two special quadrupoles

```text
QFF1, QDF1
```

and used half-strength bends. The forward dispersion suppressor is

```text
QF,   D1, BH, D2,
QD,   D1, BH, D2,
QFF1, D1, BH, D2,
QDF1, D1, BH, D2
```


In [5]:
QFF1() = Quadrupole(L=L_quad, Kn1=kQFF1)
QDF1() = Quadrupole(L=L_quad, Kn1=kQDF1)

function build_DISPSUPF()
    return Beamline(
        [
            QF_arc(),  D1(), BH(), D2(),
            QD_arc(),  D1(), BH(), D2(),
            QFF1(),    D1(), BH(), D2(),
            QDF1(),    D1(), BH(), D2(),
        ],
        species_ref=species_ref,
        E_ref=E_ref,
    )
end

dispsupF = build_DISPSUPF()


Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1              Quadrupole   0
  2              Drift        0.5
  3              SBend        1.109
  4              Drift        7.969
  5              Quadrupole   9.21
  6              Drift        9.71
  7              SBend        10.319
  8              Drift        17.179
  9              Quadrupole   18.42
  10             Drift        18.92
  11             SBend        19.529
  12             Drift        26.389
  13             Quadrupole   27.63
  14             Drift        28.13
  15             SBend        28.739
  16             Drift        35.599


## 3.5 Track one particle and compute a linear map with GTPSA

To compute Twiss propagation, we need the linear transfer matrix of a beamline.

We get it as the Jacobian of final phase-space coordinates with respect to initial phase-space coordinates, using `AutoGTPSA()`.


In [6]:
function track_a_particle(v0, beamline)
    v = similar(v0)
    v .= v0

    b = Bunch(v; species=beamline.species_ref, p_over_q_ref=beamline.p_over_q_ref)
    track!(b, beamline)

    # Return owned storage; exposing the Bunch's internal GTPSA array can
    # leave DifferentiationInterface holding invalid native-memory references.
    return copy(b.coords.v)
end

function transfer_matrix_gtpsa(beamline; x0=zeros(6))
    prep = DI.prepare_jacobian(
        track_a_particle,
        AutoGTPSA(),
        x0,
        Constant(beamline),
    )

    return DI.jacobian(
        track_a_particle,
        prep,
        AutoGTPSA(),
        x0,
        Constant(beamline),
    )
end

function linear_map_with_descriptor(beamline, d; x0=zeros(6))
    xvars = vars(d)
    v0 = [x0[i] + xvars[i] for i in 1:6]
    vout = track_a_particle(v0, beamline)

    M = Matrix{Any}(undef, 6, 6)
    for i in 1:6, j in 1:6
        powers = zeros(Int, 6)
        powers[j] = 1
        M[i, j] = par(vout[i], [powers...,:])
    end
    return M
end

parameter_gradient(x) = GTPSA.gradient(x, include_params=true)[7:end]
tps_const(x) = try x[zeros(Int, 6)] catch; x end

function transverse_blocks(J)
    Mx = J[1:2, 1:2]
    My = J[3:4, 3:4]
    return Mx, My
end


transverse_blocks (generic function with 1 method)

## 3.6 Twiss helpers

A transverse transfer matrix propagates the Twiss matrix by

$$
\Sigma_2 = M \Sigma_1 M^T,
$$

where

$$
\Sigma =
\begin{pmatrix}
\beta & -\alpha \\
-\alpha & \gamma
\end{pmatrix},
\qquad
\gamma = \frac{1+\alpha^2}{\beta}.
$$


In [7]:
struct Twiss
    beta
    alpha
end

gamma(t::Twiss) = (1 + t.alpha^2) / t.beta

function sigma_matrix(t::Twiss)
    return [
         t.beta    -t.alpha
        -t.alpha    gamma(t)
    ]
end

function twiss_from_sigma(S)
    return Twiss(S[1, 1], -S[1, 2])
end

function propagate_twiss(t::Twiss, M)
    S2 = M * sigma_matrix(t) * transpose(M)
    return twiss_from_sigma(S2)
end

function periodic_twiss_from_matrix(M)
    tr = M[1, 1] + M[2, 2]
    cosμ = tr / 2

    if abs(cosμ) >= 1
        error("Unstable one-cell matrix: |Tr(M)/2| >= 1")
    end

    sinμ = sqrt(1 - cosμ^2)

    # Choose the sign of sin(mu) so that beta is positive.
    if M[1, 2] / sinμ < 0
        sinμ = -sinμ
    end

    beta = M[1, 2] / sinμ
    alpha = (M[1, 1] - M[2, 2]) / (2sinμ)

    return Twiss(beta, alpha)
end


periodic_twiss_from_matrix (generic function with 1 method)

## 3.7 Compute the Twiss at the end of `DISPSUPF`

First compute the periodic arc Twiss from the forward arc FODO cell. Then propagate those Twiss parameters through `DISPSUPF`.

The result is the input Twiss for the Chapter 3 matching section.


In [8]:
J_arc = transfer_matrix_gtpsa(fodoAF)
Mx_arc, My_arc = transverse_blocks(J_arc)

arc_a = periodic_twiss_from_matrix(Mx_arc)
arc_b = periodic_twiss_from_matrix(My_arc)

println("Periodic arc Twiss at the entrance of DISPSUPF:")
@printf("beta.a  = %.10f, alpha.a = %.10f\n", arc_a.beta, arc_a.alpha)
@printf("beta.b  = %.10f, alpha.b = %.10f\n", arc_b.beta, arc_b.alpha)

J_dispsup = transfer_matrix_gtpsa(dispsupF)
Mx_dispsup, My_dispsup = transverse_blocks(J_dispsup)

input_a = propagate_twiss(arc_a, Mx_dispsup)
input_b = propagate_twiss(arc_b, My_dispsup)

println("\nTwiss at the end of DISPSUPF:")
@printf("beta.a  = %.10f, alpha.a = %.10f\n", input_a.beta, input_a.alpha)
@printf("beta.b  = %.10f, alpha.b = %.10f\n", input_b.beta, input_b.alpha)


Periodic arc Twiss at the entrance of DISPSUPF:
beta.a  = 30.6233206758, alpha.a = -2.4014114074
beta.b  = 5.5491714814, alpha.b = 0.4766827694

Twiss at the end of DISPSUPF:
beta.a  = 30.8327398954, alpha.a = -2.4223022200
beta.b  = 5.3367028145, alpha.b = 0.4620065960


## 3.8 Define the Chapter 3 matching quadrupoles

The Chapter 3 matching section adds four new quadrupoles:

```text
QFF2, QDF2, QFF3, QDF3
```

Their initial strengths are set to the straight-section FODO strength.


In [9]:
K_ss = 0.351957452649287

K_start = [
     K_ss,   # QFF2
    -K_ss,   # QDF2
     K_ss,   # QFF3
    -K_ss,   # QDF3
]

QFF2(k) = Quadrupole(L=L_quad, Kn1=k)
QDF2(k) = Quadrupole(L=L_quad, Kn1=k)
QFF3(k) = Quadrupole(L=L_quad, Kn1=k)
QDF3(k) = Quadrupole(L=L_quad, Kn1=k)

println("Initial Chapter 3 strengths:")
@printf("QFF2 Kn1 = %.15f\n", K_start[1])
@printf("QDF2 Kn1 = %.15f\n", K_start[2])
@printf("QFF3 Kn1 = %.15f\n", K_start[3])
@printf("QDF3 Kn1 = %.15f\n", K_start[4])


Initial Chapter 3 strengths:
QFF2 Kn1 = 0.351957452649287
QDF2 Kn1 = -0.351957452649287
QFF3 Kn1 = 0.351957452649287
QDF3 Kn1 = -0.351957452649287


## 3.9 Build the Chapter 3 matching section `MSSF`

The forward matching section is

```text
QFF2, D1, DB, D2,
QDF2, D1, DB, D2,
QFF3, D1, DB, D2,
QDF3, D1, DB, D2
```


In [10]:
function build_MSSF(k)
    return Beamline(
        [
            QFF2(k[1]), D1(), DB(), D2(),
            QDF2(k[2]), D1(), DB(), D2(),
            QFF3(k[3]), D1(), DB(), D2(),
            QDF3(k[4]), D1(), DB(), D2(),
        ],
        species_ref=species_ref,
        E_ref=E_ref,
    )
end

mssf0 = build_MSSF(K_start)


Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1              Quadrupole   0
  2              Drift        0.5
  3              Drift        1.109
  4              Drift        6.964
  5              Quadrupole   8.205
  6              Drift        8.705
  7              Drift        9.314
  8              Drift        15.169
  9              Quadrupole   16.41
  10             Drift        16.91
  11             Drift        17.519
  12             Drift        23.374
  13             Quadrupole   24.615
  14             Drift        25.115
  15             Drift        25.724
  16             Drift        31.579


## 3.10 Build the combined line `ARC_TO_SSF`

For the matching calculation, the varying part is `MSSF`. The full conceptual line is

```text
ARC_TO_SSF = DISPSUPF + MSSF
```


In [11]:
function build_ARC_TO_SSF(k)
    # Depending on the exact Beamline interface, you may prefer a dedicated
    # concatenation method. This explicit form keeps the element ordering clear.
    return Beamline(
        [
            QF_arc(),  D1(), BH(), D2(),
            QD_arc(),  D1(), BH(), D2(),
            QFF1(),    D1(), BH(), D2(),
            QDF1(),    D1(), BH(), D2(),

            QFF2(k[1]), D1(), DB(), D2(),
            QDF2(k[2]), D1(), DB(), D2(),
            QFF3(k[3]), D1(), DB(), D2(),
            QDF3(k[4]), D1(), DB(), D2(),
        ],
        species_ref=species_ref,
        E_ref=E_ref,
    )
end

arc_to_ssf0 = build_ARC_TO_SSF(K_start)


Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         s [m]  
  1              Quadrupole   0
  2              Drift        0.5
  3              SBend        1.109
  4              Drift        7.969
  5              Quadrupole   9.21
  6              Drift        9.71
  7              SBend        10.319
  8              Drift        17.179
  9              Quadrupole   18.42
  10             Drift        18.92
  11             SBend        19.529
  12             Drift        26.389
  13             Quadrupole   27.63
  14             Drift        28.13
  15             SBend        28.739
  16             Drift        35.599
  17             Quadrupole   36.84
  18             Drift        37.34
  19             Drift        37.949
  ⋮       ⋮      ⋮            ⋮

                       13 rows omitted

## 3.11 Set the straight-section target Twiss

The target values are the periodic Twiss parameters of the forward straight-section FODO cell.


In [12]:
target_a = Twiss(27.20598820, -2.40249037)
target_b = Twiss(4.96091411,   0.48460556)

println("Straight-section target Twiss:")
@printf("beta.a  = %.10f, alpha.a = %.10f\n", target_a.beta, target_a.alpha)
@printf("beta.b  = %.10f, alpha.b = %.10f\n", target_b.beta, target_b.alpha)


Straight-section target Twiss:
beta.a  = 27.2059882000, alpha.a = -2.4024903700
beta.b  = 4.9609141100, alpha.b = 0.4846055600


## 3.12 Define the matching residual

For a trial vector

```julia
K = [K_QFF2, K_QDF2, K_QFF3, K_QDF3]
```

we build `MSSF`, compute its transfer matrix with GTPSA, propagate the Twiss parameters from the end of `DISPSUPF`, and compare the result with the straight-section target.


In [13]:
function end_twiss_after_MSSF(k)
    mssf = build_MSSF(k)

    J = transfer_matrix_gtpsa(mssf)
    Mx, My = transverse_blocks(J)

    end_a = propagate_twiss(input_a, Mx)
    end_b = propagate_twiss(input_b, My)

    return end_a, end_b
end

const d_mssf_knobs = Descriptor(6, 2, 4, 1)
const dk_mssf = params(d_mssf_knobs)

function end_twiss_after_MSSF_with_knobs(k)
    mssf = build_MSSF(k .+ dk_mssf)

    J = linear_map_with_descriptor(mssf, d_mssf_knobs)
    Mx, My = transverse_blocks(J)

    end_a = propagate_twiss(input_a, Mx)
    end_b = propagate_twiss(input_b, My)

    return end_a, end_b
end

function matching_residual_with_knobs(k)
    end_a, end_b = end_twiss_after_MSSF_with_knobs(k)

    return [
        (end_a.beta  - target_a.beta) / target_a.beta,
         end_a.alpha - target_a.alpha,
        (end_b.beta  - target_b.beta) / target_b.beta,
         end_b.alpha - target_b.alpha,
    ]
end

function matching_residual(k)
    return tps_const.(matching_residual_with_knobs(k))
end

function merit(k)
    r = matching_residual(k)
    return sum(abs2, r)
end

println("Initial residual:")
println(matching_residual(K_start))
println("Initial merit = ", merit(K_start))


Initial residual:
[0.1333071112419315, -0.019811849968438544, 0.07574989128724273, -0.022598964030221158]
Initial merit = 0.02441205451210894


## 3.13 Compute the residual Jacobian

The optimizer needs the Jacobian

$$
J_{ij} =
\frac{\partial r_i}{\partial K_j},
$$

where

```text
r_i = one of the four Twiss residuals
K_j = one of the four matching quadrupole strengths
```

We compute this Jacobian with a single GTPSA descriptor that contains both roles needed by the calculation:

```text
Descriptor(6, 2, 4, 1)
```

The six ordinary GTPSA variables represent the initial phase-space coordinates used to extract the linear transfer matrix. The four descriptor parameters represent the small changes in `QFF2`, `QDF2`, `QFF3`, and `QDF3`. After propagating the Twiss functions through this parameter-dependent matrix, the residual constants give the usual residual vector and `GTPSA.gradient(..., include_params=true)[7:end]` gives the four derivatives in each Jacobian row.


In [14]:
function residual_jacobian(k)
    residual = matching_residual_with_knobs(k)
    return vcat((parameter_gradient(r)' for r in residual)...)
end

J0 = residual_jacobian(K_start)

println("Initial residual Jacobian:")
display(J0)


Initial residual Jacobian:


4×4 Matrix{Float64}:
  0.282404    2.0138   -8.57766    -3.48022
 14.963      -4.67538   7.33067     8.6164
 -0.26902   -15.6738   -0.0204385  14.0364
 -2.73836    -7.16412   2.24938     4.39803

## 3.14 Optimize the four strengths

Use a damped Gauss-Newton step:

$$
(J^T J + \lambda I)\Delta K = -J^T r.
$$

Then update

$$
K \leftarrow K + \Delta K.
$$


In [15]:
function match_with_residual_jacobian(k0; maxiter=40, tol=1e-12, λ0=1e-3)
    k = copy(k0)
    λ = λ0
    history = NamedTuple[]

    for iter in 1:maxiter
        r = matching_residual(k)
        J = residual_jacobian(k)
        merit_now = sum(abs2, r)

        A = transpose(J) * J + λ * I
        step = -(A \ (transpose(J) * r))

        trial = k + step
        merit_trial = merit(trial)

        if merit_trial < merit_now
            k = trial
            λ /= 10
        else
            λ *= 10
        end

        push!(history, (
            iter = iter,
            merit = merit_now,
            step_norm = norm(step),
            lambda = λ,
        ))

        if merit_now < tol || norm(step) < tol
            break
        end
    end

    return k, history
end

K_match, history = match_with_residual_jacobian(K_start)

println("Optimization history:")
for h in history
    @printf("%3d  merit = %.6e  step = %.3e  lambda = %.3e\n",
            h.iter, h.merit, h.step_norm, h.lambda)
end

println("\nOptimized Chapter 3 strengths:")
@printf("QFF2 Kn1 = %.15f\n", K_match[1])
@printf("QDF2 Kn1 = %.15f\n", K_match[2])
@printf("QFF3 Kn1 = %.15f\n", K_match[3])
@printf("QDF3 Kn1 = %.15f\n", K_match[4])

println("\nFinal residual:")
println(matching_residual(K_match))
println("Final merit = ", merit(K_match))


Optimization history:
  1  merit = 2.441205e-02  step = 1.816e-02  lambda = 1.000e-02
  2  merit = 2.441205e-02  step = 1.816e-02  lambda = 1.000e-01
  3  merit = 2.441205e-02  step = 1.812e-02  lambda = 1.000e+00
  4  merit = 2.441205e-02  step = 1.783e-02  lambda = 1.000e+01
  5  merit = 2.441205e-02  step = 1.575e-02  lambda = 1.000e+02
  6  merit = 2.441205e-02  step = 7.538e-03  lambda = 1.000e+01
  7  merit = 1.047305e-02  step = 1.280e-02  lambda = 1.000e+00
  8  merit = 4.495030e-03  step = 7.949e-03  lambda = 1.000e-01
  9  merit = 1.898997e-05  step = 1.333e-03  lambda = 1.000e-02
 10  merit = 2.397425e-08  step = 1.364e-04  lambda = 1.000e-03
 11  merit = 1.746293e-12  step = 3.454e-07  lambda = 1.000e-04
 12  merit = 1.501343e-19  step = 5.104e-10  lambda = 1.000e-05

Optimized Chapter 3 strengths:
QFF2 Kn1 = 0.356272039609577
QDF2 Kn1 = -0.346603509526460
QFF3 Kn1 = 0.378810944804757
QDF3 Kn1 = -0.351626495212383

Final residual:
[-1.3842086792967628e-14, -3.41948691584548

## 3.15 Check the matched Twiss parameters


In [16]:
matched_a, matched_b = end_twiss_after_MSSF(K_match)

@printf("%-10s %18s %18s %18s\n", "quantity", "target", "matched", "difference")
@printf("%-10s %18.10e %18.10e %18.10e\n", "beta.a",  target_a.beta,  matched_a.beta,  matched_a.beta  - target_a.beta)
@printf("%-10s %18.10e %18.10e %18.10e\n", "alpha.a", target_a.alpha, matched_a.alpha, matched_a.alpha - target_a.alpha)
@printf("%-10s %18.10e %18.10e %18.10e\n", "beta.b",  target_b.beta,  matched_b.beta,  matched_b.beta  - target_b.beta)
@printf("%-10s %18.10e %18.10e %18.10e\n", "alpha.b", target_b.alpha, matched_b.alpha, matched_b.alpha - target_b.alpha)


quantity               target            matched         difference
beta.a       2.7205988200e+01   2.7205988200e+01  -3.7658764995e-13
alpha.a     -2.4024903700e+00  -2.4024903700e+00  -3.4194869158e-14
beta.b       4.9609141100e+00   4.9609141100e+00   1.1013412404e-13
alpha.b      4.8460556000e-01   4.8460556000e-01  -5.4567461660e-14


## 3.16 Export the Chapter 3 strengths

Save the optimized matching strengths in a small Julia file for later chapters.


In [17]:
solution_text = """
# chapter3_mSSF_solution.jl
# Auto-generated by Chapter 3 notebook.

K_QFF2 = $(repr(K_match[1]))
K_QDF2 = $(repr(K_match[2]))
K_QFF3 = $(repr(K_match[3]))
K_QDF3 = $(repr(K_match[4]))
"""

solution_path = joinpath("lattices", "chapter_3", "chapter3_mSSF_solution.jl")
mkpath(dirname(solution_path))
write(solution_path, solution_text)

println("Wrote: ", solution_path)
println()
println(solution_text)


Wrote: lattices\chapter_3\chapter3_mSSF_solution.jl

# chapter3_mSSF_solution.jl
# Auto-generated by Chapter 3 notebook.

K_QFF2 = 0.35627203960957676
K_QDF2 = -0.34660350952645963
K_QFF3 = 0.3788109448047574
K_QDF3 = -0.3516264952123834



## Exercises

### Reverse matching section

Construct the reverse matching section using the same procedure. Reverse the drift ordering and optimize the four reverse matching quadrupoles:

```text
QFR2, QDR2, QFR3, QDR3
```

Save the optimized strengths as

```text
lattices/chapter_3/chapter3_mSSR_solution.jl
```

Compare the reverse matching strengths with the forward matching strengths.

### Merit weight

Restart from the initial strengths and modify the merit function by putting a large weight on the first beta residual. For example:

```julia
weights = [1e10, 1, 1, 1]
```

Then use

```julia
sum(weights .* abs2.(matching_residual(k)))
```

in the matching loop and compare the final residuals with the equal-weight case.
